In [ ]:
### installed
# pip install --upgrade pip
# pip install ipykernel

# pip install --upgrade ipywidgets jupyter

# pip install datasets pandas tqdm spacy nltk
# pip install transformers
# python -m spacy download en_core_web_sm
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


# pip install pytorch-lightning
# pip install rouge-score
# pip install torch-optimizer

### not installed

In [ ]:
# Dataset:
#     WikiHow sep-style examples if possible

# Teacher model:
#     TransformerSum WikiHow extractive model
#     preferably roberta-base-ext-sum or distilroberta-base-ext-sum

# Labeling:
#     get sentence importance scores
#     choose top ceil(n * ratio)
#     save scores + labels

# Ratios:
#     0.20, 0.30, 0.40

# Saved format:
#     JSONL

# Later QNLP:
#     load JSONL
#     simplify/prune sentences
#     parse with DepCCG
#     encode with IQPAnsatz
#     train sentence classifier

In [1]:
import re
import math
import json
import math
import torch
import random
import statistics
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from tqdm.notebook import tqdm

from extractive import ExtractiveSummarizer

# print(torch.cuda.is_available())

In [ ]:
model = ExtractiveSummarizer.load_from_checkpoint("Models/epoch%3D3.ckpt")
model.eval()
model.freeze()
model.cpu()
# model.cuda()
print(next(model.parameters()).device)

In [ ]:
MULTI_NEWLINES = re.compile(r"\n+")
MULTI_SPACES = re.compile(r"\s+")
STRAY_QUOTES = re.compile(r'"\s*')

def clean_text(text: str) -> str:
    # Replace newlines/tabs with spaces
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # Remove strange quote leftovers
    # text = STRAY_QUOTES.sub('"', text)

    # Collapse repeated whitespace
    text = MULTI_SPACES.sub(" ", text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

def valid_article_length(article: str, min_n_sentences=6) -> bool:
    return len(article.split(". ")) >= min_n_sentences

def label_with_top_ratio(sentences_any, top_ratio: float=0.3) -> list[dict]:
    if type(sentences_any) is str:
        sentences_list = sentences_any.split(". ")
    elif type(sentences_any) is list:
        sentences_list = sentences_any
    else:
        raise ValueError("Input must be a string or list of sentences")
    
    raw_predictions = model.predict_sentences(
        input_sentences=sentences_list,
        raw_scores=True,
        tokenized=False
    )

    scored = [
        {
            "sentence_id": i,
            "sentence": sentences_list[i],
            "score": float(item[1])
        }
        for i, item in enumerate(raw_predictions)
    ]

    n = len(scored)
    k = max(1, math.ceil(n * top_ratio))

    top_indices = sorted(
        range(n),
        key=lambda i: scored[i]["score"],
        reverse=True
    )[:k]

    selected = set(top_indices)

    for item in scored:
        item["label"] = item["sentence_id"] in selected
        item["label_vector"] = [0, 1] if item["label"] else [1, 0]

    return scored

# slower than z-score method
def label_with_random_ratio(sentences_any, min_ratio=0.15, max_ratio=0.30):
    if type(sentences_any) is str:
        sentences_list = sentences_any.split(". ")
    elif type(sentences_any) is list:
        sentences_list = sentences_any
    else:
        raise ValueError("Input must be a string or list of sentences")
    raw_predictions = model.predict_sentences(
        input_sentences=sentences_list,
        raw_scores=True,
        tokenized=False
    )

    scored = [
        {
            "sentence_id": i,
            "sentence": sentences_list[i],
            "score": float(item[1])
        }
        for i, item in enumerate(raw_predictions)
    ]

    ratio = random.uniform(min_ratio, max_ratio)
    n = len(scored)
    k = max(1, math.ceil(n * ratio))

    top_indices = sorted(
        range(n),
        key=lambda i: scored[i]["score"],
        reverse=True
    )[:k]

    selected = set(top_indices)

    for item in scored:

        is_positive = item["sentence_id"] in selected

        item["label"] = is_positive
        item["label_vector"] = [0, 1] if is_positive else [1, 0]

    return scored

def label_with_zscore(sentences_any, z_threshold=0.5, min_positive=1, max_positive=5) -> list[dict]:

    if type(sentences_any) is str:
        sentences_list = sentences_any.split(". ")
    elif type(sentences_any) is list:
        sentences_list = sentences_any
    else:
        raise ValueError("Input must be a string or list of sentences")

    raw_predictions = model.predict_sentences(
        input_sentences=sentences_list,
        raw_scores=True,
        tokenized=False
    )

    scored = [
        {
            "sentence_id": i,
            "sentence": sentences_list[i],
            "score": float(item[1])
        }
        for i, item in enumerate(raw_predictions)
    ]

    scores = [item["score"] for item in scored]

    mean_score = statistics.mean(scores)

    # Avoid std=0 crash
    std_score = statistics.pstdev(scores)

    # If all scores identical
    if std_score == 0:
        std_score = 1e-8

    threshold = mean_score + z_threshold * std_score

    selected = set()

    for item in scored:
        if item["score"] >= threshold:
            selected.add(item["sentence_id"])

    # Ensure minimum positives
    if len(selected) < min_positive:

        ranked = sorted(
            range(len(scored)),
            key=lambda i: scored[i]["score"],
            reverse=True
        )

        selected = set(ranked[:min_positive])

    # Limit maximum positives
    if len(selected) > max_positive:

        ranked_selected = sorted(
            selected,
            key=lambda i: scored[i]["score"],
            reverse=True
        )

        selected = set(ranked_selected[:max_positive])

    for item in scored:

        is_positive = item["sentence_id"] in selected

        item["label"] = is_positive
        item["label_vector"] = [0, 1] if is_positive else [1, 0]

    return scored

def label_articles(articles: list[str]):
    labeled_articles = []
    for article_sentences_str in tqdm(articles, desc="Labeling articles"):
        if valid_article_length(article_sentences_str, 7):
            a =label_with_top_ratio(article_sentences_str, top_ratio=0.40)
            # a = label_with_zscore(article_sentences_str, 
            #             z_threshold=0.5, min_positive=1, max_positive=4)
            # a = label_with_random_ratio(article_sentences_str)
            labeled_articles.append(a)
    return labeled_articles

def count_invalid_articles(articles):
    invalid_count = 0
    valid, invalid = [], []
    invalid_idx = []
    for i, a in enumerate(articles):
        if not valid_article_length(a):
            invalid_count += 1
            invalid.append(len(a.split(". ")))
            invalid_idx.append(i)
            continue
        valid.append(len(a.split(". ")))

    print(invalid_count)
    print(invalid)
    print(invalid_idx)
    # print(valid)


In [ ]:
dataset = pd.read_csv('Dataset/Raw/WikiHow/wikihowSep.csv')
articles_all = dataset['text'].dropna().astype(str).apply(clean_text).tolist()
print(len(articles_all))

In [ ]:
a = articles_all.copy() 
# np.random.shuffle(a)
articles_to_label = a[:20000]
articles_scored = label_articles(articles_to_label)

# 1:08:19 | 200_000 | random_ratio()
# 54:39   | 200_000 | z-score()

In [ ]:
good_scores = []
krc = []
n_sentences = 0
n_ = 0
article_no_good_scores = 0
for k, article in enumerate(articles_scored):
    n_sentences += len(article)
    before = len(good_scores)
    for i, sent_dict in enumerate(article):
        # if sent_dict["score"] > 0.30:
        if sent_dict["label"]:
            good_scores.append(sent_dict["score"])
    
    krc.append(len(good_scores) - before)
    if before == len(good_scores):
        article_no_good_scores += 1

average_n_sentences = np.mean([len(s) for s in articles_scored])
print(f"n_articles: {len(articles_scored)} | n_sentences: {n_sentences} | average sents/article: {average_n_sentences:0.2f} | n_sentences_valid_score: {len(good_scores)} | average score: {np.mean(good_scores):0.2f}")
print(f"article_no_good_scores: {article_no_good_scores} | Valid score ratio: {(len(good_scores) / n_sentences):0.2f}")
print(f"1: {krc.count(1)} | 2: {krc.count(2)} | 3: {krc.count(3)} | 4: {krc.count(4)} | 5: {krc.count(5)} | 6: {krc.count(6)} | 7: {krc.count(7)} | 8: {krc.count(8)} | 9: {krc.count(9)} | 10: {krc.count(10)} | 11: {krc.count(11)} | 12: {krc.count(12)}")

In [ ]:
#   20_000 | z-score()
# n_articles: 2710 | n_sentences: 25089 | n_sentences_valid_score: 6130 | average score: 0.26
# article_no_good_scores: 0 | Valid score ratio: 0.24 | average_n_sentences: 9.26
# 1: 530 | 2: 1215 | 3: 690 | 4: 275 | 5: 0 | 6: 0 |

#   200_000 | z-score()
# n_articles: 36881 | n_sentences: 339667 | n_sentences_valid_score: 83086 | average score: 0.26
# article_no_good_scores: 0 | Valid score ratio: 0.24 | average_n_sentences: 9.21
# 1: 7286 | 2: 16444 | 3: 9692 | 4: 3459 | 5: 0 | 6: 0 | 

In [ ]:
import json

SAVE_PATH = "Dataset/Raw/WikiHow/Labeled_wikihowSep_45ratio_20k.jsonl"

with open(SAVE_PATH, "w", encoding="utf-8") as f:

    for article_id, article_data in enumerate(articles_scored):

        record = {
            "article_id": article_id,
            "n_sentences": len(article_data),
            "n_positive": sum(1 for s in article_data if s["label"]),
            # "labeling_method": "zscore",
            # "z_threshold": 0.5,
            "labeling_method": "top_ratio",
            "top_ratio": 0.45,
            "sentences_data": article_data
        }

        json.dump(record, f, ensure_ascii=False)

        # Important: each JSON object on separate line
        f.write("\n")

In [4]:
SAVE_PATH = "Dataset/Raw/WikiHow/Labeled_wikihowSep_200k.jsonl"
dataset = []

with open(SAVE_PATH, "r", encoding="utf-8") as f:
    n_sentences = 0
    for line in f:

        record = json.loads(line)
        dataset.append(record)
        n_sentences += record["n_sentences"]

    print(f"Loaded {len(dataset)} articles | {n_sentences} sentences")

Loaded 36881 articles | 339667 sentences
